# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple  # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row: NamedTuple):
    if row.gender == "female":
        yield (row.age, row)


def REDUCE(age: str, rows: Iterator[NamedTuple]):
    sum = 0
    count = 0
    for row in rows:
        sum += row.social_contacts
        count += 1
    if count > 0:
        yield (age, sum / count)
    else:
        yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
    id: int
    age: str
    social_contacts: int
    gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender="male", social_contacts=20),
    User(id=1, age=25, gender="female", social_contacts=240),
    User(id=2, age=25, gender="female", social_contacts=500),
    User(id=3, age=33, gender="female", social_contacts=800),
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
    return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output)  # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
    t = {}
    for k2, v2 in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(
    flatten(REDUCE(*x) for x in groupbykey(flatten(MAP(*x) for x in RECORDREADER()))),
)

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element


def groupbykey(iterable):
    t = {}
    for k2, v2 in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()


def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(
        map(
            lambda x: REDUCE(*x),
            groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))),
        )
    )

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [14]:
from typing import NamedTuple  # requires python 3.6+
from typing import Iterator


class User(NamedTuple):
    id: int
    age: str
    social_contacts: int
    gender: str


input_collection = [
    User(id=0, age=55, gender="male", social_contacts=20),
    User(id=1, age=25, gender="female", social_contacts=240),
    User(id=2, age=25, gender="female", social_contacts=500),
    User(id=3, age=33, gender="female", social_contacts=800),
]


def MAP(_, row: NamedTuple):
    if row.gender == "female":
        yield (row.age, row)


def REDUCE(age: str, rows: Iterator[NamedTuple]):
    sum = 0
    count = 0
    for row in rows:
        sum += row.social_contacts
        count += 1
    if count > 0:
        yield (age, sum / count)
    else:
        yield (age, 0)


def RECORDREADER():
    return [(u.id, u) for u in input_collection]


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5, 4))
vec = np.random.rand(4)


def MAP(coordinates: (int, int), value: int):
    i, j = coordinates
    yield (i, value * vec[j])


def REDUCE(i: int, products: Iterator[NamedTuple]):
    sum = 0
    for p in products:
        sum += p
    yield (i, sum)


def RECORDREADER():
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            yield ((i, j), mat[i, j])


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.426973589862511)),
 (1, np.float64(1.426973589862511)),
 (2, np.float64(1.426973589862511)),
 (3, np.float64(1.426973589862511)),
 (4, np.float64(1.426973589862511))]

## Inverted index 

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]


def RECORDREADER():
    for docid, document in enumerate(documents):
        yield ("{}".format(docid), document)


def MAP(docId: str, body: str):
    for word in set(body.split(" ")):
        yield (word, docId)


def REDUCE(word: str, docIds: Iterator[str]):
    yield (word, sorted(docIds))


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]


def RECORDREADER():
    for docid, document in enumerate(documents):
        for lineid, line in enumerate(document.split("\n")):
            yield ("{}:{}".format(docid, lineid), line)


def MAP(docId: str, line: str):
    for word in line.split(" "):
        yield (word, 1)


def REDUCE(word: str, counts: Iterator[int]):
    sum = 0
    for c in counts:
        sum += c
    yield (word, sum)


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element


def groupbykey(iterable):
    t = {}
    for k2, v2 in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()


def groupbykey_distributed(map_partitions, PARTITIONER):
    global reducers
    partitions = [dict() for _ in range(reducers)]
    for map_partition in map_partitions:
        for k2, v2 in map_partition:
            p = partitions[PARTITIONER(k2)]
            p[k2] = p.get(k2, []) + [v2]
    return [
        (partition_id, sorted(partition.items(), key=lambda x: x[0]))
        for (partition_id, partition) in enumerate(partitions)
    ]


def PARTITIONER(obj):
    global reducers
    return hash(obj) % reducers


def MapReduceDistributed(
    INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None
):
    map_partitions = map(
        lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)),
        INPUTFORMAT(),
    )
    if COMBINER != None:
        map_partitions = map(
            lambda map_partition: flatten(
                map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))
            ),
            map_partitions,
        )
    reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)  # shuffle
    reduce_outputs = map(
        lambda reduce_partition: (
            reduce_partition[0],
            flatten(
                map(
                    lambda reduce_input_group: REDUCE(*reduce_input_group),
                    reduce_partition[1],
                )
            ),
        ),
        reduce_partitions,
    )

    print(
        "{} key-value pairs were sent over a network.".format(
            sum(
                [
                    len(vs)
                    for (k, vs) in flatten(
                        [partition for (partition_id, partition) in reduce_partitions]
                    )
                ]
            )
        )
    )
    return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2


def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for docid, document in enumerate(split):
            for lineid, line in enumerate(document.split("\n")):
                yield ("{}:{}".format(docid, lineid), line)

    split_size = int(np.ceil(len(documents) / maps))
    for i in range(0, len(documents), split_size):
        yield RECORDREADER(documents[i : i + split_size])


def MAP(docId: str, line: str):
    for word in line.split(" "):
        yield (word, 1)


def REDUCE(word: str, counts: Iterator[int]):
    sum = 0
    for c in counts:
        sum += c
    yield (word, sum)


# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [
    (partition_id, list(partition)) for (partition_id, partition) in partitioned_output
]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('what', 10)]),
 (1, [('it', 18)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0


def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for value in split:
            yield (value, None)

    split_size = int(np.ceil(len(input_values) / maps))
    for i in range(0, len(input_values), split_size):
        yield RECORDREADER(input_values[i : i + split_size])


def MAP(value: int, _):
    yield (value, None)


def PARTITIONER(key):
    global reducers
    global max_value
    global min_value
    bucket_size = (max_value - min_value) / reducers
    bucket_id = 0
    while (key > (bucket_id + 1) * bucket_size) and (
        (bucket_id + 1) * bucket_size < max_value
    ):
        bucket_id += 1
    return bucket_id


def REDUCE(value: int, _):
    yield (None, value)


partitioned_output = MapReduceDistributed(
    INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER
)
partitioned_output = [
    (partition_id, list(partition)) for (partition_id, partition) in partitioned_output
]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.029655333543311735)),
   (None, np.float64(0.03576559064240703)),
   (None, np.float64(0.11559521119048433)),
   (None, np.float64(0.12244607902887117)),
   (None, np.float64(0.1267520224771962)),
   (None, np.float64(0.18042104229503375)),
   (None, np.float64(0.19629300793760363)),
   (None, np.float64(0.21970466144683187)),
   (None, np.float64(0.22649228865478588)),
   (None, np.float64(0.3943389186315418)),
   (None, np.float64(0.4063534390678004)),
   (None, np.float64(0.4191161845936455)),
   (None, np.float64(0.47132010719969497)),
   (None, np.float64(0.4902759626553814)),
   (None, np.float64(0.49377128287632877))]),
 (1,
  [(None, np.float64(0.5026098652924437)),
   (None, np.float64(0.5056639881200764)),
   (None, np.float64(0.5098502562858238)),
   (None, np.float64(0.5247742799391483)),
   (None, np.float64(0.5516859646233625)),
   (None, np.float64(0.5784195775007891)),
   (None, np.float64(0.6368756265470601)),
   (None, np.float64(0.69320095

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
def RECORDREADER():
    input_values = [14, 5, 89, 23, 42, 7]
    yield from enumerate(input_values)


def MAP(k1, v1):
    yield ("max", v1)


def REDUCE(key, values):
    yield (key, max(values))


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Максимальное значение:", list(output))

Максимальное значение: [('max', 89)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
def RECORDREADER():
    input_values = [10, 20, 30, 40, 50]
    yield from enumerate(input_values)


def MAP(k1, v1):
    yield ("average", (v1, 1))


def REDUCE(key, values):
    total_sum = 0
    total_count = 0
    for val, count in values:
        total_sum += val
        total_count += count
    yield (key, total_sum / total_count)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Арифметическое среднее:", list(output))

Арифметическое среднее: [('average', 30.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
def groupbykey_sorted(iterable):
    sorted_items = sorted(iterable, key=lambda x: x[0])

    if not sorted_items:
        return []

    result = []
    current_key = sorted_items[0][0]
    current_values = []

    for key, value in sorted_items:
        if key == current_key:
            current_values.append(value)
        else:
            result.append((current_key, current_values))
            current_key = key
            current_values = [value]

    result.append((current_key, current_values))
    return result


test_data = [("a", 1), ("b", 2), ("a", 3), ("c", 4), ("b", 5)]
print("GroupByKey на основе сортировки:", groupbykey_sorted(test_data))

GroupByKey на основе сортировки: [('a', [1, 3]), ('b', [2, 5]), ('c', [4])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [24]:
def RECORDREADER():
    input_values = [1, 2, 2, 3, 4, 4, 4, 5]
    yield from enumerate(input_values)


def MAP(k1, v1):
    yield (v1, None)


def REDUCE(key, values):
    yield (key, None)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Уникальные элементы:", [k for k, v in output])

Уникальные элементы: [1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [25]:
def RECORDREADER():
    relations = [(1, 5), (12, 3), (8, 9), (15, 2)]
    yield from enumerate(relations)


def MAP(k1, v1):
    if v1[0] > 10:
        yield (v1, v1)


def REDUCE(key, values):
    yield (key, key)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Selection (Выборка):", [k for k, v in output])

Selection (Выборка): [(12, 3), (15, 2)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [26]:
def RECORDREADER():
    relations = [(1, "A", 100), (2, "B", 200), (3, "C", 300)]
    yield from enumerate(relations)


def MAP(k1, v1):
    t_prime = (v1[0], v1[1])
    yield (t_prime, t_prime)


def REDUCE(key, values):
    yield (key, key)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Projection (Проекция):", [k for k, v in output])

Projection (Проекция): [(1, 'A'), (2, 'B'), (3, 'C')]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [27]:
def RECORDREADER():
    R = [(1, "a"), (2, "b")]
    S = [(2, "b"), (3, "c")]
    yield from enumerate(R + S)


def MAP(k1, v1):
    yield (v1, v1)


def REDUCE(key, values):
    yield (key, key)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Union (Объединение):", [k for k, v in output])

Union (Объединение): [(1, 'a'), (2, 'b'), (3, 'c')]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [28]:
def RECORDREADER():
    R = [(1, "a"), (2, "b"), (3, "c")]
    S = [(2, "b"), (3, "c"), (4, "d")]
    for i, t in enumerate(R + S):
        yield (i, t)


def MAP(k1, v1):
    yield (v1, v1)


def REDUCE(key, values):
    count = sum(1 for _ in values)
    if count == 2:
        yield (key, key)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Intersection (Пересечение):", [k for k, v in output])

Intersection (Пересечение): [(2, 'b'), (3, 'c')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [29]:
def RECORDREADER():
    R = [(1, "a"), (2, "b"), (3, "c")]
    S = [(2, "b"), (4, "d")]
    for t in R:
        yield (t, "R")
    for t in S:
        yield (t, "S")


def MAP(k1, v1):
    tuple_data = k1
    relation_name = v1
    yield (tuple_data, relation_name)


def REDUCE(key, values):
    val_list = list(values)
    if val_list == ["R"]:
        yield (key, key)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Difference (Разница R - S):", [k for k, v in output])

Difference (Разница R - S): [(1, 'a'), (3, 'c')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [30]:
def RECORDREADER():
    R = [(1, 2), (3, 4)]
    S = [(2, 5), (4, 6)]
    for t in R:
        yield (t, "R")
    for t in S:
        yield (t, "S")


def MAP(k1, v1):
    t = k1
    rel = v1
    if rel == "R":
        a, b = t
        yield (b, ("R", a))
    elif rel == "S":
        b, c = t
        yield (b, ("S", c))


def REDUCE(key, values):
    b = key
    r_vals = []
    s_vals = []
    for val in values:
        rel_name = val[0]
        data = val[1]
        if rel_name == "R":
            r_vals.append(data)
        elif rel_name == "S":
            s_vals.append(data)

    for a in r_vals:
        for c in s_vals:
            yield (None, (a, b, c))


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Natural Join:", [v for k, v in output])

Natural Join: [(1, 2, 5), (3, 4, 6)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
def RECORDREADER():
    relations = [("group1", 10, "c1"), ("group1", 20, "c2"), ("group2", 50, "c3")]
    yield from enumerate(relations)


def MAP(k1, v1):
    a, b, c = v1
    yield (a, b)


def REDUCE(key, values):
    total = sum(values)
    yield (key, total)


output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Grouping and Aggregation:", list(output))

Grouping and Aggregation: [('group1', 30), ('group2', 50)]


# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [32]:
I = 5
J = 4
M_mat = np.random.rand(I, J)
V_vec = np.random.rand(J)


def RR_PHASE_1():
    for i in range(I):
        for j in range(J):
            yield (j, ("M", i, M_mat[i, j]))
    for j in range(J):
        yield (j, ("V", V_vec[j]))


def MAP_PHASE_1(k1, v1):
    yield (k1, v1)


def REDUCE_PHASE_1(key, values):
    v_val = 0
    m_elements = []

    for val in values:
        if val[0] == "V":
            v_val = val[1]
        elif val[0] == "M":
            m_elements.append((val[1], val[2]))

    for i, m_val in m_elements:
        yield (i, m_val * v_val)


def RR_PHASE_2(phase_1_output):
    for k, v in phase_1_output:
        yield (k, v)


def MAP_PHASE_2(k1, v1):
    yield (k1, v1)


def REDUCE_PHASE_2(key, values):
    yield (key, sum(values))


phase_1_out = list(MapReduce(RR_PHASE_1, MAP_PHASE_1, REDUCE_PHASE_1))
output_vector = list(
    MapReduce(lambda: RR_PHASE_2(phase_1_out), MAP_PHASE_2, REDUCE_PHASE_2)
)

reference_solution = np.matmul(M_mat, V_vec)
result_vec = np.zeros(I)
for i, val in output_vector:
    result_vec[i] = val

print(
    "Большой вектор (Out-of-Core) обработан корректно:",
    np.allclose(reference_solution, result_vec),
)

Большой вектор (Out-of-Core) обработан корректно: True


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [33]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        yield from iterable


def groupbykey(iterable):
    t = {}
    for k2, v2 in iterable:
        t[k2] = [*t.get(k2, []), v2]
    return t.items()


def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(
        REDUCE(*x) for x in groupbykey(flatten(MAP(*x) for x in RECORDREADER()))
    )

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [34]:
I = 2
J = 3
K = 4 * 10
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)


def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])


def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(I):
        yield ((i, k), small_mat[i, j] * w)


def REDUCE(key, values):
    (i, k) = key
    total_sum = sum(values)
    yield (key, total_sum)

Проверьте своё решение

In [35]:
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)


def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    I = max(i for ((i, k), vw) in reduce_output) + 1
    K = max(k for ((i, k), vw) in reduce_output) + 1
    mat = np.empty(shape=(I, K))
    for (i, k), vw in reduce_output:
        mat[i, k] = vw
    return mat


np.allclose(reference_solution, asmatrix(solution))

True

In [36]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i, k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [37]:
I = 2
J = 3
K = 4
M_mat = np.random.rand(I, J)
N_mat = np.random.rand(J, K)


def RECORDREADER():
    for i in range(I):
        for j in range(J):
            yield ((i, j), ("M", M_mat[i, j]))

    for j in range(J):
        for k in range(K):
            yield ((j, k), ("N", N_mat[j, k]))


def MAP(k1, v1):
    rel, matrix_val = v1

    if rel == "M":
        i, j = k1
        for k in range(K):
            yield ((i, k), ("M", j, matrix_val))
    else:
        j, k = k1
        for i in range(I):
            yield ((i, k), ("N", j, matrix_val))


def REDUCE(key, values):
    m_vals = {}
    n_vals = {}

    for val in values:
        rel, j, matrix_val = val
        if rel == "M":
            m_vals[j] = matrix_val
        else:
            n_vals[j] = matrix_val

    total = sum(m_vals.get(j, 0) * n_vals.get(j, 0) for j in range(J))
    yield (key, total)


output = list(MapReduce(RECORDREADER, MAP, REDUCE))
reference_solution = np.matmul(M_mat, N_mat)
print("Матрицы совпадают (обе из RR)?", np.allclose(reference_solution, asmatrix(output)))

Матрицы совпадают (обе из RR)? True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [38]:
def INPUTFORMAT():
    def RECORDREADER_M():
        for i in range(I):
            for j in range(J):
                yield ((i, j), ("M", M_mat[i, j]))

    def RECORDREADER_N():
        for j in range(J):
            for k in range(K):
                yield ((j, k), ("N", N_mat[j, k]))

    yield RECORDREADER_M()
    yield RECORDREADER_N()


reducers = 4

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)

flat_output = [
    item for partition_id, partition_data in partitioned_output for item in partition_data
]

print(
    "Distributed (отдельные RR) совпадает?",
    np.allclose(np.matmul(M_mat, N_mat), asmatrix(flat_output)),
)

48 key-value pairs were sent over a network.
Distributed (отдельные RR) совпадает? True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [39]:
import random


def INPUTFORMAT_RANDOM():
    all_elements = []
    for i in range(I):
        for j in range(J):
            all_elements.append(((i, j), ("M", M_mat[i, j])))
    for j in range(J):
        for k in range(K):
            all_elements.append(((j, k), ("N", N_mat[j, k])))

    random.shuffle(all_elements)

    splits = 3
    split_size = len(all_elements) // splits + 1

    def RECORDREADER_SPLIT(split_data):
        yield from split_data

    for s in range(0, len(all_elements), split_size):
        yield RECORDREADER_SPLIT(all_elements[s : s + split_size])


partitioned_output_random = MapReduceDistributed(
    INPUTFORMAT_RANDOM, MAP, REDUCE, COMBINER=None
)

flat_output_random = [
    item for p_id, p_data in partitioned_output_random for item in p_data
]

print(
    "Distributed (случайные подмножества) совпадает?",
    np.allclose(np.matmul(M_mat, N_mat), asmatrix(flat_output_random)),
)

48 key-value pairs were sent over a network.
Distributed (случайные подмножества) совпадает? True
